In [ ]:
# ==========================================
# CELL 1: TẢI TOÀN BỘ DỮ LIỆU & TỐI ƯU HÓA RAM
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# Đã xóa import RandomForest và SMOTE vì không dùng nữa
import warnings
warnings.filterwarnings('ignore')

print("⏳ Đang tải TOÀN BỘ dữ liệu từ file CSV (Có thể mất 1-3 phút)...")
df_ml = pd.read_csv(r"C:\Users\Admin\Desktop\DAP_Project\output\dashboard_data.csv")

df_sample = df_ml.copy()

print("🧹 Đang nén dữ liệu (Downcasting) để giải cứu 16GB RAM...")
# Kỹ thuật ép kiểu: Biến float64 thành float32, int64 thành int32 để giảm 50% dung lượng
for col in df_sample.columns:
    if df_sample[col].dtype == 'float64':
        df_sample[col] = df_sample[col].astype('float32')
    elif df_sample[col].dtype == 'int64':
        df_sample[col] = df_sample[col].astype('int32')

print(f"✅ Đã tải FULL dữ liệu thành công! Kích thước khổng lồ: {df_sample.shape}")

# Xóa bản gốc df_ml để giải phóng bộ nhớ RAM ngay lập tức
del df_ml

In [ ]:
# ==========================================
# CELL 2: DATA CLEANING & ANTI-LEAKAGE
# ==========================================
print("🧹 Đang dọn dẹp các cột gây rò rỉ dữ liệu...")

# CHỈ XÓA 3 CỘT: Year (Thiên kiến lịch sử), Distance và Duration (Rò rỉ hậu quả)
cols_to_drop = ['Year', 'Distance(mi)', 'Duration']
df_sample = df_sample.drop(columns=[c for c in cols_to_drop if c in df_sample.columns], errors='ignore')

# Xóa các dòng khuyết thiếu dữ liệu để mô hình không bị lỗi toán học
df_sample = df_sample.dropna()

print(f"✅ Đã dọn dẹp xong! Kích thước dữ liệu sẵn sàng: {df_sample.shape}")
display(df_sample.head(3)) # In thử 3 dòng ra xem

In [ ]:
# ==========================================
# CELL 3: TRAIN/TEST SPLIT
# ==========================================
print("✂️ Đang chia tách dữ liệu Huấn luyện (Train) và Kiểm thử (Test)...")

X = df_sample.drop(columns=['Severity'])
y = df_sample['Severity']

# Chia tỷ lệ 80-20, dùng stratify để giữ nguyên tỷ lệ chênh lệch của các Mức độ tai nạn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"📦 Tập Huấn luyện (Train) có: {X_train.shape[0]} dòng.")
print(f"📦 Tập Kiểm thử (Test) có: {X_test.shape[0]} dòng.")

In [ ]:
# ==========================================
# CELL 4 & 5 GỘP: XÂY DỰNG PIPELINE TỰ ĐỘNG & HUẤN LUYỆN
# ==========================================
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score

print("🛠️ Đang chế tạo các Trạm biến áp (Custom Transformers) cho Dây chuyền...")

# 1. Trạm Xử lý Nhị phân (True/False & Ngày/Đêm)
class BinaryMapper(BaseEstimator, TransformerMixin):
    def __init__(self, bool_cols, sunset_col='Sunrise_Sunset'):
        self.bool_cols = bool_cols
        self.sunset_col = sunset_col

    def fit(self, X, y=None):
        return self # Không cần "học" gì ở bước này

    def transform(self, X):
        X_copy = X.copy()
        for col in self.bool_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].astype(int)
        
        if self.sunset_col in X_copy.columns:
            mapping = {'Day': 1, 'Night': 0}
            X_copy[self.sunset_col] = X_copy[self.sunset_col].map(mapping).fillna(0).astype(int)
        return X_copy

# 2. Trạm Uốn cong Thời gian (Lượng giác hóa)
class CyclicalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, time_cols):
        self.time_cols = time_cols # Ví dụ: {'Hour': 24, 'Month': 12, 'Weekday': 7}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col, max_val in self.time_cols.items():
            if col in X_copy.columns:
                X_copy[f'{col}_sin'] = np.sin(2 * np.pi * X_copy[col] / max_val)
                X_copy[f'{col}_cos'] = np.cos(2 * np.pi * X_copy[col] / max_val)
                X_copy = X_copy.drop(columns=[col]) # Xóa cột gốc
        return X_copy

# 3. Trạm Chuyển đổi Tần suất (Frequency Encoding - CHỐNG RÒ RỈ DỮ LIỆU)
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, freq_cols):
        self.freq_cols = freq_cols
        self.freq_maps_ = {} # Bộ nhớ lưu trữ tần suất học từ tập Train

    def fit(self, X, y=None):
        # CHỈ HỌC TỪ TẬP TRAIN: Đếm tần suất và lưu vào bộ nhớ
        for col in self.freq_cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.freq_cols:
            if col in X_copy.columns:
                # Áp dụng thước đo đã học. Nếu có giá trị mới toanh ở tập Test -> điền 0
                X_copy[col] = X_copy[col].map(self.freq_maps_.get(col, {})).fillna(0)
        return X_copy

# ---------------------------------------------------------
print("🏭 Đang lắp ráp Dây chuyền hoàn chỉnh (Pipeline)...")

# Định nghĩa các cột cần xử lý
bool_cols = ['Traffic_Signal', 'Junction']
time_cols = {'Hour': 24, 'Month': 12, 'Weekday': 7}
freq_cols = ['State', 'City', 'Weather_Condition']

# LẮP RÁP: Dữ liệu sẽ đi từ trên xuống dưới
ml_pipeline = Pipeline([
    ('binary_map', BinaryMapper(bool_cols=bool_cols)),
    ('cyclical_encode', CyclicalEncoder(time_cols=time_cols)),
    ('frequency_encode', FrequencyEncoder(freq_cols=freq_cols)),
    ('classifier', lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=63,
        class_weight='balanced',
        importance_type='gain',
        n_jobs=-1,
        random_state=42
    ))
])

# ---------------------------------------------------------
print("🧠 Đang khởi động Pipeline học dữ liệu (Fit)...")
# CHỈ VỚI 1 DÒNG CODE: Vừa biến đổi data, vừa train mô hình
ml_pipeline.fit(X_train, y_train)
print("✅ Học xong! Đang dự đoán tập Test...\n")

# CHỈ VỚI 1 DÒNG CODE: Tự động chạy lại toàn bộ quy trình biến đổi cho tập Test rồi dự đoán
y_pred = ml_pipeline.predict(X_test)
y_pred_proba = ml_pipeline.predict_proba(X_test)

# ---------------------------------------------------------
print("="*50)
print("🏆 BẢNG THÀNH TÍCH PIPELINE LIGHTGBM")
print("="*50)
print(f"🎯 1. Accuracy          : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"⚖️ 2. F1-Macro          : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"📈 3. ROC-AUC (OVR)     : {roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro'):.4f}")
print("-" * 50)
print("📊 4. CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, zero_division=0))
print("="*50)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Lấy mức độ quan trọng của các tính năng
feature_imp = pd.DataFrame(sorted(zip(model_lgb.feature_importances_, X_train.columns)), columns=['Value','Feature'])

plt.figure(figsize=(10, 8))
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False))
plt.title('LightGBM Features Importance (Full Data)')
plt.tight_layout()
plt.show()